# The idealised flow against the Hawkes flow

Section 1.4.4 of the lecture notes, which sets the idealised book against the geometric one, at
the keyboard.  One path of the six-type order flow is placed three ways, from one opening
book:

* **CKS**, the idealised placement (`IdealisedFlow`): orders of one share at the touches, so that
  the book is a function of the path;
* **haw-1**, the geometric placement of the simulator, with orders of one share;
* **haw-q**, the geometric placement with lognormal sizes of mean $\bar m = 100$ shares and log-sd
  $0.8$, from the opening book with every size multiplied by $\bar m$.  Its sizes are read in units
  of $\bar m$.

CKS and haw-1 receive the same events and the same sizes and differ in where the orders go;
haw-1 and haw-q differ in the size of each order.  The two objects of the study are the parting
time $\chi$, the first time the mid-price of CKS and that of a Hawkes flow differ, and the
discrepancy $\mathcal D_t = P^{m,\mathrm{CKS}}_t - P^{m,\mathrm{haw}}_t$ that follows it, haw being
haw-1 or haw-q.

The flow is `config.balanced_order_flow_params()`: the resilient shape at a branching ratio of
$0.5$, direction-symmetric, with stationary intensity $\lambda^* = (1, 1, 2.5, 2.5, 1.5, 1.5)$ in
the order market buy, market sell, limit buy, limit sell, withdrawal from the bid, withdrawal from
the ask.  The bid is taken by the market sells, at $\lambda^*_2 = 1$, joined by the limit buys, at
$\lambda^*_3 = 2.5$, and drawn by the withdrawals, at $\lambda^*_5 = 1.5$, so that
$\lambda^*_3 = \lambda^*_2 + \lambda^*_5$; the ask is its mirror image.  The opening book has a spread of one tick and $L = 10$
levels a side, one on every tick, each holding $\bar S = 20$.  The notes allow any opening touches
up to $\bar S$; this notebook opens both full, $r^b(0) = r^a(0) = 1$.  Every path starts from no
past.  Prices are in ticks, so $\tau = 1$, and $\eta$ is `DepthDecay`.  The two
regimes of Corollary 1.4.39 meet at $\eta = \lambda^*_2/\lambda^*_3 = 0.4$.

Statements are cited by their numbers in the notes.  Every path and every set of marks is seeded,
and the notebook runs in about three minutes.

In [ ]:
import dataclasses
import inspect

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from unito26.lob import config
from unito26.lob.coupling import (
    CoupledFlows, Mover, OpeningTouch, PlacementStatistics, RecordedEvents, SizeUnit, VarianceRates,
    cks_first_move, cks_mids, cks_touch_walks, closed_form_rates, diffusive_scale,
    distance_terms, first_price_discrepancy, first_price_moves, fluid_emptying_time, fluid_touch,
    hawkes_mids, second_regime_rate, side_vector, touch_walks, variance_split,
)
from unito26.lob.hawkes import Clock, ExponentialHawkes
from unito26.lob.idealised import IdealisedFlow, UniformDepth
from unito26.lob.messages import BUY, SELL, GridDepth
from unito26.lob.simulate import EventType, OrderFlowSimulator
from unito26.lob.visualization import (
    INK, MUTED, PALETTE, discrepancy_fan_figure, distance_terms_figure, first_times_figure,
    mids_figure, touch_walks_figure, use_template, variance_split_figure,
)

use_template()

FLOW = config.balanced_order_flow_params()
UNIT = config.unit_mark_params()
UNCERTAIN = config.uncertain_size_mark_params()
ONE = SizeUnit(1)
MBAR = config.uncertain_size_unit()
SIZE_VARIANCE = np.expm1(UNCERTAIN.sigma_log_size ** 2)
DEPTH = UniformDepth(20)
LEVELS = GridDepth(10)
PRESSURE = np.array([event.pressure for event in EventType], dtype=float)
COUNT_COVARIANCE = FLOW.count_covariance_rate()
GROUP = {
    EventType.MARKET_BUY: "market", EventType.MARKET_SELL: "market",
    EventType.LIMIT_BUY: "limit", EventType.LIMIT_SELL: "limit",
    EventType.WITHDRAW_BUY: "withdrawal", EventType.WITHDRAW_SELL: "withdrawal",
}


def opening(depth):
    """Both touches full, one tick apart: the touches of the opening book, in units."""
    return {BUY: (1000, depth), SELL: (1001, depth)}


OPENING = opening(DEPTH)
FULL = OpeningTouch(1.0)
OPENING_MID = 1000.5


def with_decay(marks, depth_decay):
    """The same marks, at eta = depth_decay."""
    return dataclasses.replace(marks, depth_decay=depth_decay)


def drawn(seed, horizon):
    """A path of the balanced flow on (0, horizon], from no past."""
    return RecordedEvents.drawn(ExponentialHawkes(FLOW, rng=seed), horizon)


def time_of(events, event):
    """The time of an event of a path, NaN for None."""
    return np.nan if event is None else float(events.times[event])


def source(function):
    print(inspect.getsource(function))

## 1. Three flows on one path

Definition 1.4.25, Assumption 1.4.26 with equation (1.4.23), and Remark 1.4.27.  The side vectors
$\varpi^b = \mathbf e_3 - \mathbf e_2 - \mathbf e_5$ and $\varpi^a = \mathbf e_4 - \mathbf e_1 - \mathbf e_6$
say what an order of one share does to the size resting on each side, and
$\varpi^b - \varpi^a = \varpi$.  The package counts types from zero, so
$\mathbf e_3$ is entry 2.  Balance is $(\varpi^b)^\top\lambda^* = (\varpi^a)^\top\lambda^* = 0$.

In [ ]:
stationary = FLOW.stationary_intensity()
print("lambda*                    ", stationary.round(6))
print("varpi^b                    ", side_vector(BUY))
print("varpi^a                    ", side_vector(SELL))
print("varpi^b - varpi^a = varpi: ", np.array_equal(side_vector(BUY) - side_vector(SELL), PRESSURE))
print(f"(varpi^b)' lambda* = {side_vector(BUY) @ stationary:.1e},  "
      f"(varpi^a)' lambda* = {side_vector(SELL) @ stationary:.1e}")
print(f"branching ratio {FLOW.branching_ratio:.3f}")

One path of 600 s, folded into CKS and into each Hawkes book by `CoupledFlows.on`, which records
both books after every event.  The events are drawn once, as a `RecordedEvents`, and each book
reads its own replay of them.  The Hawkes books draw their marks on a generator of their own, so
that the path is common and only the placement differs.

In [ ]:
SEED, HORIZON = 4, 600.0

events = drawn(SEED, HORIZON)
pairs = {
    "haw-1": CoupledFlows.on(events, UNIT, ONE, DEPTH, OPENING, LEVELS, SEED),
    "haw-q": CoupledFlows.on(events, UNCERTAIN, MBAR, DEPTH, OPENING, LEVELS, SEED),
}
print(f"{len(events)} events on (0, {HORIZON:g}] s;  eta = {UNIT.depth_decay}")
for flow, pair in pairs.items():
    parting = pair.parting
    print(f"{flow}: reached the horizon {pair.breach is None};  chi = {parting.time:.2f} s, "
          f"a {parting.event_type.name.lower()}, moved by {parting.moved.value}")

What each placement makes of an event.  The CKS placement reads the book and draws nothing:

In [ ]:
source(IdealisedFlow._place)

The geometric placement draws an offset $\zeta_n$ for a limit order, which it prices $1 + \zeta_n$
ticks from the opposite touch, and, for a withdrawal, a level, in proportion to the sizes resting
on the side:

In [ ]:
source(OrderFlowSimulator._limit_price)
source(OrderFlowSimulator._withdrawal)

At a spread of one tick a geometric limit order joins its touch when $\zeta_n = 0$, with
probability $\eta$, and rests behind it otherwise.  A CKS limit order joins its touch unless the
touch holds $\bar S$, and then trades a share at the opposite touch.

The three mid-prices on the path, with $\chi$ of each Hawkes flow marked:

In [ ]:
mids_figure(pairs, "The mid-price of the three books on one path, η = 0.2")

The first minute:

In [ ]:
figure = mids_figure(pairs, "The first minute")
figure.update_xaxes(range=[0.0, 60.0])
figure.update_yaxes(range=[998.5, 1003.5])
figure

On this path haw-q parts from CKS at 19.4 s and haw-1 at 23.4 s, each at a market order that moves
its own best price first.  CKS moves its mid for the first time at about 48 s, and less often than
the Hawkes books throughout.  The three mids follow the same path in the large, since the events
are the same, and differ by a few ticks.

The touch walks of equation (1.4.24), $\Upsilon^b = S^{b,1}_0 + (\varpi^b)^\top N$ and
$\Upsilon^a = S^{a,1}_0 + (\varpi^a)^\top N$, count what arrives at each side; here both opening
touches are $\bar S$.  With orders of one share a withdrawal from an occupied side removes
one share wherever it is drawn, so by equation (1.4.23) the whole bid of haw-1 is
$S^b_0(\tau\mathbb N) + (\varpi^b)^\top N(t) = \Upsilon^b(t) + (L-1)\bar S$, and its whole-book
imbalance, bid less ask, is $S^{b,1}_0 - S^{a,1}_0 + \varpi^\top N(t)$, whatever the marks.  The CKS book has the same whole-book imbalance.
Both statements are exact; here they are checked at every event of the path, on the whole books
that `CoupledFlows` records, a column per price with the size at a bid and minus the size at an
ask.

In [ ]:
def sides(depth):
    """The size on the whole bid and on the whole ask after each event."""
    values = depth.to_numpy()
    return np.where(values > 0, values, 0.0).sum(axis=1), np.where(values < 0, -values, 0.0).sum(axis=1)


walk_bid = OPENING[BUY][1] + np.cumsum(side_vector(BUY)[events.types])
walk_ask = OPENING[SELL][1] + np.cumsum(side_vector(SELL)[events.types])
imbalance = OPENING[BUY][1] - OPENING[SELL][1] + np.cumsum(PRESSURE[events.types])

bid, ask = sides(pairs["haw-1"].hawkes_depth)
cks_bid, cks_ask = sides(pairs["haw-1"].cks_depth)
print("haw-1: bid = Upsilon^b + (L - 1) Sbar at every event:", np.array_equal(bid, walk_bid + (LEVELS - 1) * DEPTH))
print("haw-1: ask = Upsilon^a + (L - 1) Sbar at every event:", np.array_equal(ask, walk_ask + (LEVELS - 1) * DEPTH))
print("haw-1: bid - ask = S^b_0 - S^a_0 + varpi' N, every event:", np.array_equal(bid - ask, imbalance))
print("CKS:   bid - ask = S^b_0 - S^a_0 + varpi' N, every event:", np.array_equal(cks_bid - cks_ask, imbalance))

In [ ]:
figure = go.Figure()
for name, walk, colour in (("Υ<sup>b</sup>", walk_bid, PALETTE[0]), ("Υ<sup>a</sup>", walk_ask, PALETTE[1])):
    figure.add_scatter(x=events.times, y=walk, mode="lines", line_shape="hv", name=name,
                       line=dict(color=colour, width=1.5))
figure.add_hline(y=DEPTH, line=dict(color=MUTED, width=1, dash="dot"))
figure.update_layout(
    title="The touch walks on the path; the dotted line is S̄", height=380,
    xaxis_title="time (s)", yaxis_title="shares", legend=dict(orientation="h", y=1.12, x=0),
)
figure

Under balance the walks have no drift, and they spread as $\sqrt{(\varpi^b)^\top D\,\varpi^b\,t}$, with $D$
the count covariance rate.  A side of $L\bar S$ shares therefore lasts a time of the order of
$(L\bar S)^2/(\varpi^b)^\top D\,\varpi^b$, and the assumption that no side empties is a statement about
horizons short against it.

In [ ]:
bid_variance = side_vector(BUY) @ COUNT_COVARIANCE @ side_vector(BUY)
print(f"(varpi^b)' D varpi^b = {bid_variance:.4f} shares² a second")
print(f"(L Sbar)^2 / (varpi^b)' D varpi^b = {(LEVELS * DEPTH) ** 2 / bid_variance:.0f} s")

## 2. When the mid-prices part

Definitions 1.4.28 and 1.4.34, Lemma 1.4.29, Corollaries 1.4.30 and 1.4.33, Propositions
1.4.35 to 1.4.37, Corollary 1.4.39 and the paragraphs that follow it, for CKS against haw-1.

A fan of 240 paths of 400 s, each placed by CKS and by haw-1 at ten values of $\eta$, the mark
seed of a path being the same at every $\eta$.  `first_price_discrepancy` folds the two books alone
and stops at $\chi$.  It returns a `Parting`: the event, its time and type, the side of the best
price that differs, and the book that moved it.

In [ ]:
FAN_PATHS, FAN_HORIZON = 240, 400.0
ETAS = (0.1, 0.2, 0.3, 0.35, 0.4, 0.45, 0.5, 0.6, 0.7, 1.0)

fan = [drawn(1000 + k, FAN_HORIZON) for k in range(FAN_PATHS)]
partings = {
    eta: [first_price_discrepancy(path, with_decay(UNIT, eta), ONE, DEPTH, OPENING, LEVELS, k)
          for k, path in enumerate(fan)]
    for eta in ETAS
}


def parting_record(record):
    """chi, which book moved, and the kind of event, over the paths that parted."""
    parted = [parting for parting in record if parting is not None]
    chi = np.array([parting.time for parting in parted])
    return {
        "parted": len(parted),
        "mean chi (s)": chi.mean(),
        "SE": chi.std(ddof=1) / np.sqrt(len(chi)),
        **{f"{mover.value} moved": np.mean([parting.moved is mover for parting in parted]) for mover in Mover},
        **{f"at a {kind}": np.mean([GROUP[parting.event_type] == kind for parting in parted])
           for kind in ("market", "limit", "withdrawal")},
    }


print(pd.DataFrame({eta: parting_record(record) for eta, record in partings.items()}).T
      .rename_axis("eta").round(3).to_string())

Every path parts within 400 s at every $\eta$, and at $\chi$ only one book has moved: the column
`both moved` is zero throughout.  Below the threshold haw-1 moves first on most paths at
$\eta = 0.1$, 78%, and less often as $\eta$ grows.  Above it CKS moves first on every path from
$\eta = 0.6$, and the mean of $\chi$ no longer depends on $\eta$: it is $\mathbb E\psi^{\mathrm{CKS}}$,
41.3 s, a function of the path.  The change is gradual: haw-1 still moves first on 12% of the
paths at $\eta = 0.4$ and on 4% at $0.45$.

### The quiet book

By Corollary 1.4.33, while the spread is one tick and the four touches hold two shares or more, no
event moves a best price: the first move of each book is the first event that empties one of its
touches.  `touch_walks` folds each book alone to its first move of Definition 1.4.34,
$\psi^{\mathrm{CKS}}$ and $\psi^{\mathrm{haw}\text{-}1}$, and records its touches up to it.  On the path of section 1:

In [ ]:
walks = touch_walks(events, UNIT, ONE, DEPTH, OPENING, LEVELS, SEED)
figure = touch_walks_figure(walks, "haw-1", "The touches of CKS and of haw-1 to the first move of each, η = 0.2")
figure.add_hline(y=1, line=dict(color=MUTED, width=1, dash="dot"), row="all", col=1)
figure

Across the fan, at each $\eta$ of the first regime: the touch a book empties at its first move
held one share before it, and $\chi = \psi^{\mathrm{CKS}} \wedge \psi^{\mathrm{haw}\text{-}1}$ unless the two
books made the same move at that event.

In [ ]:
def emptied_from_one_share(walks, prefix):
    """Whether, on the row before the first move of a book, one of its touches held one share."""
    sizes = walks[[f"{prefix}BidSize1", f"{prefix}AskSize1"]].to_numpy()
    moved = np.flatnonzero(np.isnan(sizes[:, 0]))
    return moved.size == 0 or sizes[moved[0] - 1].min() == 1


rows = {}
first_moves = {}
for eta in (0.1, 0.2, 0.3, 0.35, 0.4):
    marks = with_decay(UNIT, eta)
    first_moves[eta] = [first_price_moves(path, marks, ONE, DEPTH, OPENING, LEVELS, k) for k, path in enumerate(fan)]
    first = np.array([np.nanmin([time_of(path, moves.cks), time_of(path, moves.hawkes)])
                      for path, moves in zip(fan, first_moves[eta])])
    chi = np.array([parting.time for parting in partings[eta]])
    quiet = [emptied_from_one_share(touch_walks(path, marks, ONE, DEPTH, OPENING, LEVELS, k), prefix)
             for k, path in enumerate(fan[:60]) for prefix in ("CKS", "Haw")]
    rows[eta] = {
        "chi = min(psi) on": f"{np.sum(chi == first)}/{FAN_PATHS}",
        "the same move first on": f"{np.sum(chi > first)}/{FAN_PATHS}",
        "first moves from one share": f"{sum(quiet)}/{len(quiet)}",
    }
print(pd.DataFrame(rows).T.rename_axis("eta").to_string())

The last column is read on the first 60 paths, both books.

### The CKS touches from the counts

Proposition 1.4.35: before $\psi^{\mathrm{CKS}}$ each CKS touch is its walk less
$\bar\Upsilon - \bar S$, where $\bar\Upsilon(t) = \max\big(\bar S, \max_{s \le t}\max(\Upsilon^b(s), \Upsilon^a(s))\big)$:
a limit order at a full touch trades a share at the opposite one instead of joining its own, and
$\bar\Upsilon - \bar S$ counts those.  So $\psi^{\mathrm{CKS}}$ is the first time one walk falls
$\bar S$ below $\bar\Upsilon$, a function of the path computed without a book.

In [ ]:
source(cks_first_move)

In [ ]:
counted = cks_touch_walks(events, DEPTH, OPENING)
folded = walks.loc[:counted.index[-1]]

figure = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.08,
                       subplot_titles=("The touch walks and their running maximum", "The CKS touches"))
for column, name, colour in (("BidTouchWalk", "Υ<sup>b</sup>", PALETTE[0]), ("AskTouchWalk", "Υ<sup>a</sup>", PALETTE[1]),
                             ("RunningMaximum", "running maximum", INK)):
    figure.add_scatter(x=counted.index, y=counted[column], mode="lines", line_shape="hv", name=name,
                       line=dict(color=colour, width=1.5), row=1, col=1)
for column, name, colour in (("CKSBidSize1", "best bid, folded", PALETTE[0]), ("CKSAskSize1", "best ask, folded", PALETTE[1])):
    figure.add_scatter(x=folded.index, y=folded[column], mode="lines", line_shape="hv", name=name,
                       line=dict(color=colour, width=3), opacity=0.35, row=2, col=1)
    figure.add_scatter(x=counted.index, y=counted[column], mode="lines", line_shape="hv",
                       name=name.replace("folded", "from the counts"), line=dict(color=colour, width=1.5, dash="dot"),
                       row=2, col=1)
figure.update_xaxes(title_text="time (s)", row=2, col=1)
figure.update_yaxes(title_text="shares")
figure.update_layout(title="Proposition 1.4.35 on the path of section 1", height=620, legend=dict(orientation="h", y=-0.15, x=0))
figure

In [ ]:
agree = [cks_first_move(path, DEPTH, OPENING) == moves.cks for path, moves in zip(fan, first_moves[0.2])]
print(f"psi^CKS from the counts is the first move of the folded book on {sum(agree)} of {FAN_PATHS} paths")

### The two regimes

The touch walks of CKS have no drift under balance, and by Proposition 1.4.35 CKS first moves
by a drawdown of the order of $\bar S$, in a time of the order of $\bar S^2$.  The touch of
haw-1 drifts by an amount of order one a second, and below the threshold it empties in a time
of the order of $\bar S$.  At a fixed $\bar S$ either book may move first; as $\bar S$ grows
the two scales separate, and the notes let $\bar S \to \infty$.

Proposition 1.4.37: as $\bar S$ grows, with each opening touch over $\bar S$ tending to $r^x(0)$,
the touch of the side $x$ of haw-1 at $\bar S s$, over $\bar S$, tends to the fluid touch $r^x(s)$,
the solution of equation (1.4.26) with $L_x = L - 1 + r^x(0)$.  From the full opening both sides
follow

$$r' = \eta\lambda^*_3 - \lambda^*_2 - \frac{\lambda^*_5}{L}\,r, \qquad r(0) = 1,$$

that is $r(s) = r_\infty + (1 - r_\infty)e^{-\lambda^*_5 s/L}$ with
$r_\infty = L(\eta\lambda^*_3 - \lambda^*_2)/\lambda^*_5$.  By Proposition 1.4.36 a limit order joins the touch
with probability $\eta$, a market order takes a share from it, and a withdrawal draws it in proportion
to its size, the side holding $L\bar S$ to first order.  By Corollary 1.4.39, if
$\eta\lambda^*_3 < \lambda^*_2$ the fluid touch reaches zero at the time of equation (1.4.28),

$$s^* = \frac{L}{\lambda^*_5}\log\Big(1 + \frac{\lambda^*_5}{L(\lambda^*_2 - \eta\lambda^*_3)}\Big),$$

and $\chi/\bar S \to s^*$, haw-1 moving first.  If $\eta\lambda^*_3 \ge \lambda^*_2$ it stays positive,
and $\chi/\bar S \to \infty$.

In [ ]:
print(pd.DataFrame({
    eta: {"r_inf": fluid_touch(FLOW, with_decay(UNIT, eta), LEVELS, FULL, np.inf),
          "s*": fluid_emptying_time(FLOW, with_decay(UNIT, eta), LEVELS, FULL),
          "Sbar s* (s)": DEPTH * fluid_emptying_time(FLOW, with_decay(UNIT, eta), LEVELS, FULL)}
    for eta in ETAS
}).T.rename_axis("eta").round(4).to_string())

The survival of $\chi$ at four values of $\eta$, two either side of the threshold, with that of
$\psi^{\mathrm{CKS}}$ beside it.  The dotted step is the fluid prediction, $\chi = \bar S s^*$; in the
second regime there is none, and $\chi$ is $\psi^{\mathrm{CKS}}$.

In [ ]:
SURVIVAL_ETAS = (0.2, 0.35, 0.45, 0.7)
psi_cks = np.array([time_of(path, cks_first_move(path, DEPTH, OPENING)) for path in fan])
clock = np.geomspace(1.0, FAN_HORIZON, 300)

times = {"S̄ = 20": {}}
predicted = {"S̄ = 20": {}}
for eta in SURVIVAL_ETAS:
    label = f"η = {eta}"
    times["S̄ = 20"][label] = {
        "haw-1": [np.nan if parting is None else parting.time for parting in partings[eta]],
        "CKS": psi_cks,
    }
    s_star = fluid_emptying_time(FLOW, with_decay(UNIT, eta), LEVELS, FULL)
    if np.isfinite(s_star):
        predicted["S̄ = 20"][label] = {"fluid limit, S̄ s*": (clock < DEPTH * s_star).astype(float)}

first_times_figure(times, clock, predicted, "P(χ > t); for CKS, P(ψ<sup>CKS</sup> > t)",
                   "The parting time of haw-1, and the first move of CKS")

At $\eta = 0.2$ the survival of $\chi$ falls before that of $\psi^{\mathrm{CKS}}$ and is nearly
spent by $\bar S s^* = 35$ s.  At $0.35$ the fluid time, 105 s, lies beyond most of
$\psi^{\mathrm{CKS}}$: at $\bar S = 20$ the two scales, $\bar S s^*$ and $\bar S^2$, are not yet apart,
and $\chi$ follows $\psi^{\mathrm{CKS}}$ with haw-1 cutting its tail.  Above the threshold the two
curves coincide.  The race as $\bar S$ grows is measured at the end of this section.

The fluid limit itself.  The fluid touch at six values of $\eta$, and the touch of haw-1 on one
path at $\bar S = 160$, over $\bar S$, at $\eta = 0.2$ and $0.7$, up to its first move:

In [ ]:
LARGE = UniformDepth(160)
scaled = np.linspace(0.0, 8.0, 401)

figure = go.Figure()
for slot, eta in enumerate((0.1, 0.2, 0.3, 0.35, 0.5, 0.7)):
    figure.add_scatter(x=scaled, y=fluid_touch(FLOW, with_decay(UNIT, eta), LEVELS, FULL, scaled), mode="lines",
                       name=f"r(s), η = {eta}", line=dict(color=PALETTE[slot % len(PALETTE)], width=2))
large_path = drawn(5, 8.0 * LARGE)
for eta, colour in ((0.2, PALETTE[1]), (0.7, PALETTE[5])):
    walk = touch_walks(large_path, with_decay(UNIT, eta), ONE, LARGE, opening(LARGE), LEVELS, 5)
    figure.add_scatter(x=walk.index / LARGE, y=walk["HawBidSize1"] / LARGE, mode="lines", line_shape="hv",
                       name=f"haw-1 best bid / S̄, S̄ = 160, η = {eta}", line=dict(color=colour, width=1), opacity=0.8)
figure.add_hline(y=0.0, line=dict(color=MUTED, width=1))
figure.update_layout(title="The fluid touch, and the touch of haw-1 at S̄ = 160", height=440,
                     xaxis_title="s = t / S̄", yaxis_title="touch / S̄", yaxis_range=[-0.2, 5.5])
figure

$\psi^{\mathrm{haw}\text{-}1}/\bar S$ against $s^*$ as $\bar S$ grows, at three values of $\eta$ in the first
regime.  A fan of 48 paths, long enough for the largest case, each cut at $2.5\,\bar S s^*$ and
folded from the opening book at $\bar S$; the first moves are read by `first_price_moves`, which
also says whether haw-1 moved before CKS.

In [ ]:
SWEEP_PATHS, SWEEP_DEPTHS, SWEEP_ETAS, CUT = 48, (20, 40, 80, 160), (0.1, 0.2, 0.3), 2.5
s_star = {eta: fluid_emptying_time(FLOW, with_decay(UNIT, eta), LEVELS, FULL) for eta in SWEEP_ETAS}
SWEEP_HORIZON = CUT * max(SWEEP_DEPTHS) * max(s_star.values())
sweep = [drawn(2000 + k, SWEEP_HORIZON) for k in range(SWEEP_PATHS)]
print(f"{SWEEP_PATHS} paths of {SWEEP_HORIZON:.0f} s")


def scaled_first_moves(eta, depth):
    """psi^haw-1 / Sbar on each path of the sweep, NaN where haw-1 did not move before the cut,
    and whether haw-1 moved before CKS."""
    first, haw_first = [], []
    for k, path in enumerate(sweep):
        cut = path.head(int(np.searchsorted(path.times, CUT * depth * s_star[eta], side="right")))
        moves = first_price_moves(cut, with_decay(UNIT, eta), ONE, UniformDepth(depth), opening(depth), LEVELS, k)
        first.append(time_of(cut, moves.hawkes) / depth)
        haw_first.append(moves.hawkes is not None and (moves.cks is None or moves.hawkes < moves.cks))
    return np.array(first), np.array(haw_first)


sweep_moves = {(eta, depth): scaled_first_moves(eta, depth) for eta in SWEEP_ETAS for depth in SWEEP_DEPTHS}
scaled_psi = {key: psi for key, (psi, _) in sweep_moves.items()}
haw_first = {key: first for key, (_, first) in sweep_moves.items()}
print(pd.DataFrame({
    key: {"s*": s_star[key[0]], "mean psi / Sbar": np.nanmean(value),
          "SE": np.nanstd(value, ddof=1) / np.sqrt(np.isfinite(value).sum()),
          "censored": int(np.isnan(value).sum())}
    for key, value in scaled_psi.items()
}).T.rename_axis(["eta", "Sbar"]).round(3).to_string())

In [ ]:
figure = go.Figure()
for slot, eta in enumerate(SWEEP_ETAS):
    colour = PALETTE[slot]
    means = [np.nanmean(scaled_psi[eta, depth]) for depth in SWEEP_DEPTHS]
    errors = [np.nanstd(scaled_psi[eta, depth], ddof=1) / np.sqrt(SWEEP_PATHS) for depth in SWEEP_DEPTHS]
    figure.add_scatter(x=SWEEP_DEPTHS, y=means, error_y=dict(array=errors, color=colour), mode="lines+markers",
                       name=f"E ψ<sup>haw-1</sup> / S̄, η = {eta}", line=dict(color=colour, width=2))
    figure.add_hline(y=s_star[eta], line=dict(color=colour, width=1.5, dash="dot"),
                     annotation_text=f"s* = {s_star[eta]:.3f}", annotation_position="top left")
figure.update_xaxes(type="log", tickvals=SWEEP_DEPTHS, title_text="S̄")
figure.update_layout(title="The first move of haw-1, over S̄, against s*", height=440,
                     yaxis_title="ψ<sup>haw-1</sup> / S̄", legend=dict(orientation="h", y=-0.2, x=0))
figure

No path is censored.  $\psi^{\mathrm{haw}\text{-}1}/\bar S$ approaches $s^*$ from below, slowly: at
$\bar S = 160$ it is 4% short at $\eta = 0.1$, 7% at $0.2$ and 12% at $0.3$.  The gap grows as the
slope at which the fluid touch reaches zero, $\eta\lambda^*_3 - \lambda^*_2$, flattens: $-0.75$,
$-0.5$ and $-0.25$.

### The second regime

The paragraphs after Corollary 1.4.39, which the notes give as heuristics.  Above the threshold the fluid touch settles at $r_\infty > 0$.  On a side of
finite size the touch of haw-1 relaxes towards $(r_\infty/L)$ times the size resting on the whole
side, which by equation (1.4.23) is $L\bar S$ plus a walk without drift, and fluctuates about that centre
by $O(\sqrt{\bar S})$.  haw-1 therefore moves by a large deviation of its touch about the centre,
or when its side is nearly exhausted, on the scale $(L\bar S)^2/(\varpi^b)^\top D\,\varpi^b$: $L^2$ times the
diffusive scale $\bar S^2/(\varpi^b)^\top D\,\varpi^b$ on which CKS moves.  On the path of section 1, at
$\eta = 0.7$:

In [ ]:
ABOVE = with_decay(UNIT, 0.7)
above = CoupledFlows.on(events, ABOVE, ONE, DEPTH, OPENING, LEVELS, SEED)
whole_bid, whole_ask = sides(above.hawkes_depth)
centre = fluid_touch(FLOW, ABOVE, LEVELS, FULL, np.inf) / LEVELS
moments = above.frame["TimeStamp"]

figure = go.Figure()
for side, touch, whole, colour in (("bid", "HawBidSize1", whole_bid, PALETTE[0]), ("ask", "HawAskSize1", whole_ask, PALETTE[1])):
    figure.add_scatter(x=moments, y=above.frame[touch], mode="lines", line_shape="hv", name=f"best {side}",
                       line=dict(color=colour, width=1.5))
    figure.add_scatter(x=moments, y=centre * whole, mode="lines", line_shape="hv",
                       name=f"(r<sub>∞</sub>/L) × the whole {side}", line=dict(color=colour, width=1.5, dash="dot"))
figure.update_layout(title="The touches of haw-1 and their centres, η = 0.7", height=400,
                     xaxis_title="time (s)", yaxis_title="shares", legend=dict(orientation="h", y=-0.28, x=0))
figure

In [ ]:
scale = diffusive_scale(FLOW, DEPTH)
moves = int((np.diff(above.frame["HawBidPrice1"]) != 0).sum() + (np.diff(above.frame["HawAskPrice1"]) != 0).sum())
print(f"r_inf / L = {centre:.2f};  diffusive scale {scale:.0f} s;  the life of a side, L^2 times it, {LEVELS ** 2 * scale:.0f} s")
print(f"moves of a best price of haw-1 on the path: {moves}")

Both touches open at $\bar S = 20$ and climb towards their centres, half of their sides here,
on the scale of the relaxation time $L\bar S/\lambda^*_5 \approx 133$ s; after that each follows its centre,
which moves with the whole side.  No best price of haw-1 moves on the 600 s of the path.

On a book of many levels, then, CKS moves first.  Both first moves are on the scale $\bar S^2$,
and their order is set by $L$, not by $\bar S$: $\chi = \psi^{\mathrm{CKS}}$, a function of the path.
Heuristically $\psi^{\mathrm{CKS}}/\bar S^2$ is close in law to the first time one of two correlated
Brownian motions, started at zero by the full opening, falls one unit below the running maximum of
both, and the ratio of $\mathbb E\psi^{\mathrm{CKS}}$ to the diffusive scale is close to the mean of
that time, a constant of the specification.  $\psi^{\mathrm{CKS}}$ is read from the counts, on the
fan above at $\bar S = 10$ and $20$, and on the sweep at $\bar S = 40$.

In [ ]:
def cks_first_move_times(paths, depth):
    return np.array([time_of(path, cks_first_move(path, UniformDepth(depth), opening(depth))) for path in paths])


cks_rows = {}
for depth, paths in ((10, fan), (20, fan), (40, sweep)):
    psi = cks_first_move_times(paths, depth)
    scale = diffusive_scale(FLOW, UniformDepth(depth))
    cks_rows[depth] = {"paths": len(paths), "censored": int(np.isnan(psi).sum()),
                   "scale (s)": scale, "E psi^CKS (s)": np.nanmean(psi),
                   "E psi^CKS / scale": np.nanmean(psi) / scale,
                   "SE": np.nanstd(psi, ddof=1) / scale / np.sqrt(np.isfinite(psi).sum())}
cks_ratio = {depth: row["E psi^CKS / scale"] for depth, row in cks_rows.items()}
print(pd.DataFrame(cks_rows).T.rename_axis("Sbar").round(3).to_string())

The ratio reads 0.29, 0.31 and 0.34 at $\bar S = 10$, 20 and 40: about 0.3 for the full
opening, the differences within one and a half standard errors.

### The race as $\bar S$ grows

haw-1 first moves near $\bar S s^*$, in the first regime, and CKS near
$k\,\bar S^2/(\varpi^b)^\top D\,\varpi^b$, with $k$ the ratio just measured.  The two times meet at

$$\bar S_\times = \frac{s^*\,(\varpi^b)^\top D\,\varpi^b}{k},$$

below which CKS should move first on most paths, and above which haw-1 should.  On the sweep, the
share of the paths on which haw-1 moves before CKS, with its standard error:

In [ ]:
k = np.mean(list(cks_ratio.values()))
crossing = {eta: s_star[eta] * bid_variance / k for eta in SWEEP_ETAS}
print(f"k = {k:.3f};  Sbar_x = " + ",  ".join(f"{crossing[eta]:.1f} at eta = {eta}" for eta in SWEEP_ETAS))
print(pd.DataFrame({
    key: {"haw-1 first": first.mean(), "SE": np.sqrt(first.mean() * (1 - first.mean()) / first.size)}
    for key, first in haw_first.items()
}).T.rename_axis(["eta", "Sbar"]).round(3).to_string())

In [ ]:
depths = np.geomspace(min(cks_rows), max(SWEEP_DEPTHS), 50)
figure = go.Figure()
for slot, eta in enumerate(SWEEP_ETAS):
    colour = PALETTE[slot]
    figure.add_scatter(x=SWEEP_DEPTHS, y=[depth * np.nanmean(scaled_psi[eta, depth]) for depth in SWEEP_DEPTHS],
                       mode="markers", marker=dict(color=colour, size=8), name=f"E ψ<sup>haw-1</sup>, η = {eta}")
    figure.add_scatter(x=depths, y=depths * s_star[eta], mode="lines", name=f"S̄ s*, η = {eta}",
                       line=dict(color=colour, width=1.5, dash="dot"))
colour = PALETTE[len(SWEEP_ETAS)]
figure.add_scatter(x=list(cks_rows), y=[row["E psi^CKS (s)"] for row in cks_rows.values()], mode="markers",
                   marker=dict(color=colour, size=8, symbol="square"), name="E ψ<sup>CKS</sup>")
figure.add_scatter(x=depths, y=k * depths ** 2 / bid_variance, mode="lines", name="k S̄² / (ϖ<sup>b</sup>)ᵀDϖ<sup>b</sup>",
                   line=dict(color=colour, width=1.5, dash="dot"))
figure.update_xaxes(type="log", title_text="S̄")
figure.update_yaxes(type="log", title_text="seconds")
figure.update_layout(title="The first moves of the two books: slope one against slope two", height=440,
                     legend=dict(orientation="h", y=-0.25, x=0))
figure

At every $\eta$ the share rises with $\bar S$, and it is one at $\bar S = 160$.  The one case below
its crossing depth, $\eta = 0.3$ at $\bar S = 20$, is the one in which haw-1 moves first on fewer
than half of the paths, $0.44 \pm 0.07$; at $\bar S = 20$ the share is $0.81$ at $\eta = 0.1$ and
$0.63$ at $0.2$, both above their crossings.  On the logarithmic axes the first moves of haw-1 lie on
lines of slope one, a little below $\bar S s^*$, and that of CKS on a line of slope two: the lines
cross once, and beyond the crossing the race is decided.

## 3. The discrepancy: mean and variance

Definition 1.4.40, Proposition 1.4.41 and the paragraphs that follow it, for CKS against
haw-1.  The CKS
book is a function of the path and haw-1 is not, so the variance of $\mathcal D_t$ splits into the
flow part $\mathrm{Var}\,\mathbb E[\mathcal D_t \mid \text{path}]$ and the placement part
$\mathbb E\,\mathrm{Var}[\mathcal D_t \mid \text{path}]$, the second being the variance of the
Hawkes mid given the path.

A nested fan reads the two: 120 outer paths of 600 s, each placed by haw-1 with 3 mark seeds, at
$\eta = 0.2$, in the first regime, and $\eta = 0.7$, in the second; 360 runs at each $\eta$, on one
set of paths.  The Hawkes mids are read after every event, so that a side that empties between
two times of the clock is seen.  A path is kept when no run on it empties a side.  For CKS and
haw-1 that event is a function of the path, by equation (1.4.23), so keeping a path selects on the path
alone and leaves the split of Proposition 1.4.41 *iv.* in force.

In [ ]:
OUTER, SEEDS, NESTED_HORIZON = 120, 3, 600.0
CLOCK = Clock(np.arange(0.0, NESTED_HORIZON + 1.0, 5.0))
outer = [drawn(3000 + k, NESTED_HORIZON) for k in range(OUTER)]
outer_cks = np.array([cks_mids(path, DEPTH, OPENING, LEVELS, CLOCK) for path in outer])


def nested_fan(marks, size_unit):
    """D_t on CLOCK for every outer path and mark seed, and whether the run emptied a side."""
    discrepancies = np.empty((OUTER, SEEDS, CLOCK.size))
    emptied = np.zeros((OUTER, SEEDS), dtype=bool)
    for i, path in enumerate(outer):
        for j in range(SEEDS):
            after_each = hawkes_mids(path, marks, size_unit, DEPTH, OPENING, LEVELS, Clock(path.times), 100 * i + j)
            emptied[i, j] = np.isnan(after_each).any()
            on_clock = np.r_[OPENING_MID, after_each][np.searchsorted(path.times, CLOCK, side="right")]
            discrepancies[i, j] = outer_cks[i] - on_clock
    return discrepancies, emptied


nested = {f"haw-1, η = {eta}": nested_fan(with_decay(UNIT, eta), ONE) for eta in (0.2, 0.7)}
kept = {label: ~emptied.any(axis=1) & np.isfinite(outer_cks).all(axis=1) for label, (_, emptied) in nested.items()}
for label, keep in kept.items():
    print(f"{label}: {keep.sum()} of {OUTER} paths kept")

In [ ]:
discrepancy_fan_figure({label: nested[label][0][kept[label]] for label in nested}, CLOCK, 24,
                       "D<sub>t</sub> = P<sup>m,CKS</sup><sub>t</sub> − P<sup>m,haw-1</sup><sub>t</sub> across the runs")

Proposition 1.4.41 *ii.*: the law of $\mathcal D_t$ is symmetric about zero, by the mirror symmetry
of the specification, so its mean is zero.  The standard error is that of the mean over
paths of the mean over seeds, the paths being independent and the seeds of one path not.

In [ ]:
READ_AT = (50.0, 100.0, 300.0, 600.0)
at = np.searchsorted(CLOCK, READ_AT)
splits = {label: variance_split(nested[label][0][kept[label]]) for label in nested}
print(pd.concat({
    label: split.iloc[at][["Mean", "StandardError"]].assign(z=lambda frame: frame["Mean"] / frame["StandardError"])
                .set_index(pd.Index(READ_AT, name="t (s)"))
    for label, split in splits.items()
}, axis=1).round(3).to_string())

The means lie within one and a half standard errors of zero.

Proposition 1.4.41 *iii.*: $\mathrm{Var}\,\mathcal D_t = \mathbb E\mathcal D_t^2 \ge (\tau/2)^2\,\mathbb P(\mathcal D_t \ne 0)$,
since both mids lie on the half-tick grid.  On the sample it holds run by run.  The ratio says how
close it is: within a factor of two to four at 50 s, and loose after, as $\mathcal D_t$ spreads over
several ticks.

In [ ]:
rows = {}
for label in nested:
    sample = nested[label][0][kept[label]]
    second_moment = (sample ** 2).mean(axis=(0, 1))
    differ = (sample != 0).mean(axis=(0, 1))
    for t, k in zip(READ_AT, at):
        rows[label, t] = {"E D²": second_moment[k], "P(D ≠ 0) / 4": differ[k] / 4,
                          "ratio": second_moment[k] / (differ[k] / 4)}
print(pd.DataFrame(rows).T.rename_axis(["fan", "t (s)"]).round(3).to_string())

### The variance split

`variance_split` takes the discrepancies of a nested fan, of shape (paths, seeds, times), and
returns the mean and the two parts at each time.  The mean over the $n$ seeds of a path carries
$1/n$ of the placement variance, which `FlowPart` takes back, and every variance divides by one
less than its count, so that the two parts and their sum are unbiased.  On a small sample
`FlowPart` can come out negative.

In [ ]:
source(variance_split)

In [ ]:
WINDOW = CLOCK >= 200.0
PARTS = ("Total", "FlowPart", "PlacementPart")


def rates(sample):
    """The slopes of the split between 200 s and the horizon, in ticks² a second."""
    split = variance_split(sample)
    return {part: np.polyfit(CLOCK[WINDOW], split[part].to_numpy()[WINDOW], 1)[0] for part in PARTS}


def bootstrap(sample, draws, seed):
    """The standard error of each rate, resampling the paths with their seeds."""
    rng = np.random.default_rng(seed)
    resampled = [rates(sample[rng.integers(0, len(sample), len(sample))]) for _ in range(draws)]
    return {part: np.std([draw[part] for draw in resampled], ddof=1) for part in PARTS}

Two rates to set the measured ones against.  In the second regime, on horizons short against the
life of a side, the best prices of haw-1 seldom move, $\mathcal D_t$ is close to the CKS mid less the
opening one, and by equation (1.4.3) its whole rate is the flow part $\varpi^\top D\varpi/(2\bar S)^2$:
`second_regime_rate`.  In the first regime the notes give only a first guess, which the notebook
works out under two hypotheses on the books of haw-1:

* **H1**, constant placement: an event of type $e$ is placed behind its touch with a probability
  $\alpha_e$ that depends on its type alone;
* **H2**, fixed depth: haw-1 moves as if it held $\tilde S$ on every tick it crosses,
  $2\tilde S\,(P^m_t - P^m_0)/\tau = \mathrm{OFI}_t + R_t$, with $R_t$ bounded in mean square.

An event of type $e$ then moves the CKS mid by $\varpi_e/(2\bar S)$ ticks and the haw-1 mid by
$\varpi_e(1 - \alpha_e)/(2\tilde S)$ in mean.  With $c_e = 1/(2\bar S) - (1 - \alpha_e)/(2\tilde S)$,
the difference of the two impacts, the flow part grows at $(\varpi\circ c)^\top D\,(\varpi\circ c)$,
and the placement part, the variance of the draws behind the touches, at
$\sum_e \alpha_e(1-\alpha_e)\lambda^*_e/(2\tilde S)^2$: `closed_form_rates`.  $\alpha_e$ is read as the
fraction of the events of type $e$ placed behind their touch.  When the variance of the haw-1 mid
grows without bound, H2 identifies $\tilde S$ as the slope of the regression of the order flow
imbalance of haw-1 on its mid-price; with $\Pi_T = 2(P^m_T - P^m_0)/\tau$ it is estimated as
$\mathrm{Cov}(\mathrm{OFI}_T, \Pi_T)/\mathrm{Var}\,\Pi_T$ across runs to the horizon.
`PlacementStatistics.of` reads $\alpha$ and $\tilde S$ from the books that `CoupledFlows` folds,
here the first mark seed of each outer path: 120 runs.

In [ ]:
source(closed_form_rates)

In [ ]:
def placement_statistics(marks, size_unit):
    """alpha and S~ of the Hawkes books of the first mark seed of every outer path."""
    return PlacementStatistics.of([
        CoupledFlows.on(path, marks, size_unit, DEPTH, OPENING, LEVELS, 100 * i)
        for i, path in enumerate(outer)
    ])


SECOND_REGIME = VarianceRates(second_regime_rate(FLOW, DEPTH), 0.0)
statistics = {"haw-1, η = 0.2": placement_statistics(with_decay(UNIT, 0.2), ONE)}
approximations = {
    "haw-1, η = 0.2": closed_form_rates(FLOW, DEPTH, statistics["haw-1, η = 0.2"], 0.0),
    "haw-1, η = 0.7": SECOND_REGIME,
}
for label, value in statistics.items():
    print(f"{label}: alpha = {value.behind_probability.round(3)},  S~ = {value.effective_depth:.2f}")
print(f"second-regime rate varpi' D varpi / (2 Sbar)^2 = {SECOND_REGIME.total:.5f} ticks² a second")

In [ ]:
def rate_table(labels):
    rows = {}
    for label in labels:
        sample = nested[label][0][kept[label]]
        measured, errors = rates(sample), bootstrap(sample, 200, 0)
        predicted = approximations[label]
        for part, value in zip(PARTS, (predicted.total, predicted.flow_part, predicted.placement_part)):
            rows[label, part] = {"measured": measured[part], "SE": errors[part], "closed form": value}
    return pd.DataFrame(rows).T.rename_axis(["fan", "part"])


print(rate_table(["haw-1, η = 0.2", "haw-1, η = 0.7"]).round(5).to_string())

In [ ]:
def approximation_curves(label):
    rates_of = approximations[label]
    if rates_of.placement_part == 0.0:
        return {"second-regime rate": rates_of.total * CLOCK}
    return {"closed form, total": rates_of.total * CLOCK,
            "closed form, flow part": rates_of.flow_part * CLOCK,
            "closed form, placement part": rates_of.placement_part * CLOCK}


HAW_ONE = ["haw-1, η = 0.2", "haw-1, η = 0.7"]
variance_split_figure({label: splits[label] for label in HAW_ONE}, CLOCK,
                      {label: approximation_curves(label) for label in HAW_ONE},
                      "The variance of D<sub>t</sub> and its two parts, and the closed forms")

At $\eta = 0.7$ the variance of $\mathcal D_t$ grows at $0.0041 \pm 0.0006$ ticks² a second, all of
it in the flow part, against $0.0036$ from the second-regime rate.  At $\eta = 0.2$ the total grows
at $0.0081 \pm 0.0010$ against $0.0076$ from the closed form, and the flow part at
$0.0042 \pm 0.0009$ against $0.0050$; the placement part, $0.0039 \pm 0.0006$ against $0.0027$, lies
two standard errors above.  H1 and H2 hold on the books of haw-1 only approximately, and the
split is a first guess.

### Three terms

The order flow imbalance of any book is tied to the moves of its best prices: over a window, the
imbalance less the change at the bid touch, plus the change at the ask touch, is the sum over the
moves of the best prices of the size at the outer of the two values of each, signed by the move.
For CKS that size is $\bar S$ at every move, of one tick.  Subtracting the identity for haw-1 from
that for CKS writes $2\bar S\,\mathcal D_t$ as the events that haw-1 places behind its touches,
less the mismatch of the touches at $t$, plus the mismatch of the depth through which haw-1 moves.
`distance_terms` reads the three from a `CoupledFlows`, here the haw-1 pair of section 1: the
first is the order flow imbalance of CKS less that of haw-1, the second the difference of
$S^b - S^a$, the third the outer size of each move of a best price less $\bar S$ a tick, which is
zero for every move of CKS.

In [ ]:
terms = distance_terms(pairs["haw-1"])
residual = terms["Distance"] - (terms["ImbalanceMismatch"] - terms["TouchSizeMismatch"] - terms["DepthMismatch"])
print("largest residual of the identity, over the path:", np.abs(residual).max())
print("every move of CKS is one tick onto or off a queue of Sbar:", (terms["CKSDepthMismatch"] == 0).all())
distance_terms_figure(terms, "2S̄ D<sub>t</sub> and its three terms on the path of section 1, η = 0.2")

The identity holds exactly.  On this path the third term carries the large swings of
$2\bar S\,\mathcal D_t$: the depth through which haw-1 moves is not $\bar S$ a tick.  The first term,
the events haw-1 places behind its touches, is a slow walk; the second is bounded by the sizes at
the touches.

## 4. Uncertain sizes

Proposition 1.4.41 for haw-q, and the last paragraphs of the notes' section.  haw-q receives the events of
haw-1 and places them in the same way, with sizes $\bar m\,\xi_n$, $\mathbb E\xi_n = 1$ and
$\mathrm{Var}\,\xi_n = e^{0.8^2} - 1 = 0.896$.  A market order or a withdrawal now moves a touch
whenever its size reaches the size resting there, so there is no quiet book.

The parting time first, on the fan of section 2 at one value of $\eta$ in each regime, the mark
seed of a path being the same for haw-1 and haw-q.

In [ ]:
partings_q = {
    eta: [first_price_discrepancy(path, with_decay(UNCERTAIN, eta), MBAR, DEPTH, OPENING, LEVELS, k)
          for k, path in enumerate(fan)]
    for eta in (0.2, 0.7)
}
print(pd.DataFrame({
    (flow, eta): parting_record(record)
    for flow, records in (("haw-1", partings), ("haw-q", partings_q))
    for eta, record in records.items() if eta in (0.2, 0.7)
}).T.rename_axis(["flow", "eta"]).round(3).to_string())

In [ ]:
times = {"S̄ = 20": {
    f"η = {eta}": {
        "haw-1": [np.nan if parting is None else parting.time for parting in partings[eta]],
        "haw-q": [np.nan if parting is None else parting.time for parting in partings_q[eta]],
        "CKS": psi_cks,
    }
    for eta in (0.2, 0.7)
}}
first_times_figure(times, clock, {}, "P(χ > t); for CKS, P(ψ<sup>CKS</sup> > t)",
                   "The parting time of haw-q against that of haw-1")

At $\eta = 0.2$ haw-q parts slightly earlier than haw-1, at 22.1 s on average against 23.8 s, and
moves first on 67% of the paths against 64%.  At $\eta = 0.7$ the three survival curves nearly
coincide, haw-q moving first on 2 paths of 240: its touch, near half of its side, rarely meets a
market order or a withdrawal as large as itself.

The variance.  haw-q is placed on the nested fan of section 3, on the same paths and with the same
mark seeds.  The closed form of section 3 is read on the books of haw-q, with $\alpha$ and
$\tilde S$ in units of $\bar m$.  Given the path and the earlier draws, the size $\xi_n$ of an event is
independent of whether it is placed behind its touch, so an event that reaches its touch with its
whole size enters the placement part with the variance
$\alpha_e(1-\alpha_e) + (1-\alpha_e)\,\mathrm{Var}\,\xi$.  That adds the size term
$\mathrm{Var}\,\xi\sum_e(1-\alpha_e)\lambda^*_e/(2\tilde S)^2$ to the placement part.  It sets aside
the truncation at the touch, where an order that moves the touch removes the size resting there and
not $\xi_n$, and the sizes also make random, given the path, the depth through which haw-q moves and
its whole-book imbalance: the size term is neither the whole of what the sizes add to the variance
nor a bound on it.  The table sets the split of haw-q against the closed form read on its books,
without the size term, and at $\eta = 0.7$ against the second-regime rate.  The size term is then
set against the increase from haw-1 to haw-q on the same runs.  A path is again kept when no run on
it empties a side, but for haw-q that event depends on the sizes: keeping a path selects on the
marks, and the split of the kept runs is a statement on the draws that keep the sides occupied.

In [ ]:
for eta in (0.2, 0.7):
    label = f"haw-q, η = {eta}"
    nested[label] = nested_fan(with_decay(UNCERTAIN, eta), MBAR)
    kept[label] = ~nested[label][1].any(axis=1) & np.isfinite(outer_cks).all(axis=1)
    splits[label] = variance_split(nested[label][0][kept[label]])
    print(f"{label}: {kept[label].sum()} of {OUTER} paths kept")

value = statistics["haw-q, η = 0.2"] = placement_statistics(with_decay(UNCERTAIN, 0.2), MBAR)
approximations["haw-q, η = 0.2"] = closed_form_rates(FLOW, DEPTH, value, 0.0)
approximations["haw-q, η = 0.7"] = SECOND_REGIME
size_term = (closed_form_rates(FLOW, DEPTH, value, SIZE_VARIANCE).placement_part
             - approximations["haw-q, η = 0.2"].placement_part)
print(f"haw-q, η = 0.2: alpha = {value.behind_probability.round(3)},  S~ = {value.effective_depth:.2f}")
print(f"size term at eta = 0.2: {size_term:.5f} ticks² a second")

In [ ]:
print(rate_table(["haw-1, η = 0.2", "haw-q, η = 0.2", "haw-q, η = 0.7"]).round(5).to_string())

In [ ]:
def rate_increase(label, baseline, draws, seed):
    """The rates of label less those of baseline, on the paths both keep, and the standard error
    of each difference, resampling the paths with their seeds."""
    both = kept[label] & kept[baseline]
    sample, base = nested[label][0][both], nested[baseline][0][both]

    def difference(index):
        return {part: rates(sample[index])[part] - rates(base[index])[part] for part in PARTS}

    rng = np.random.default_rng(seed)
    resampled = [difference(rng.integers(0, len(sample), len(sample))) for _ in range(draws)]
    measured = difference(np.arange(len(sample)))
    return pd.DataFrame({
        "measured": measured,
        "SE": {part: np.std([draw[part] for draw in resampled], ddof=1) for part in PARTS},
    })


print(rate_increase("haw-q, η = 0.2", "haw-1, η = 0.2", 200, 0).round(5).to_string())

In [ ]:
SIDE_BY_SIDE = ["haw-1, η = 0.2", "haw-q, η = 0.2"]
variance_split_figure({label: splits[label] for label in SIDE_BY_SIDE}, CLOCK,
                      {label: approximation_curves(label) for label in SIDE_BY_SIDE},
                      "The variance split of haw-1 and of haw-q on the same paths, η = 0.2")

In [ ]:
print(splits["haw-q, η = 0.2"].iloc[at][["Mean", "StandardError"]]
      .assign(z=lambda frame: frame["Mean"] / frame["StandardError"])
      .set_index(pd.Index(READ_AT, name="t (s)")).round(3).to_string())

At $\eta = 0.2$ the sizes raise the total rate from $0.0081$ to $0.0137$.  The closed form read on
the books of haw-q gives $0.0097$ and misses the increase, which lies in the placement part:
$0.0101 \pm 0.0014$ against $0.0033$.  On the same runs the placement part rises by
$0.0062 \pm 0.0013$ from haw-1 to haw-q, and the flow part does not change within its error,
$-0.0006 \pm 0.0011$.  The size term is $0.0094$, 2.4 standard errors above the measured increase; it sets aside the
truncation at the touch.  At $\eta = 0.7$ both flows grow at about the second-regime rate, $0.0040$ and
$0.0041$ against $0.0036$, with no placement part.  The means of $\mathcal D_t$ for haw-q lie within
two standard errors of zero.

### An empty side

The notes assume that no side of any of the three books empties on the horizon considered.  How
often a run breaks it, on the nested fan of 600 s, with the life of a side,
$(L\bar S)^2/(\varpi^b)^\top D\,\varpi^b$ for haw-1 and, with sizes,
$(L\bar S)^2/\big[(\varpi^b)^\top D\,\varpi^b + \mathrm{Var}\,\xi\,(\lambda^*_3 + \lambda^*_2 + \lambda^*_5)\big]$:

In [ ]:
market, limit, withdrawal = stationary[EventType.MARKET_SELL], stationary[EventType.LIMIT_BUY], stationary[EventType.WITHDRAW_BUY]
lifetime = {"haw-1": (LEVELS * DEPTH) ** 2 / bid_variance,
            "haw-q": (LEVELS * DEPTH) ** 2 / (bid_variance + SIZE_VARIANCE * (limit + market + withdrawal))}
print(pd.DataFrame({
    label: {"runs": emptied.size, "runs with an empty side": int(emptied.sum()),
            "fraction": emptied.mean(), "lifetime scale (s)": lifetime[label.split(",")[0]]}
    for label, (_, emptied) in nested.items()
}).T.round(4).to_string())
print(f"CKS: {int((~np.isfinite(outer_cks).all(axis=1)).sum())} of {OUTER} paths with an empty side")

And on the 48 paths of the sweep of section 2, of 1253 s each, from the opening book at
$\bar S = 20$.  By equation (1.4.23) a side of haw-1 empties exactly when its walk
$S^b_0(\tau\mathbb N) + (\varpi^b)^\top N$, here $L\bar S + (\varpi^b)^\top N$, reaches zero, which the counts decide; haw-q is folded.

In [ ]:
def haw_one_empties(path):
    opening_side = {side: (LEVELS - 1) * DEPTH + OPENING[side][1] for side in (BUY, SELL)}
    return min((opening_side[side] + np.cumsum(side_vector(side)[path.types])).min() for side in (BUY, SELL)) <= 0


def haw_q_empties(path, marks, seed):
    return np.isnan(hawkes_mids(path, marks, MBAR, DEPTH, OPENING, LEVELS, Clock(path.times), seed)).any()


long_runs = {"haw-1": np.array([haw_one_empties(path) for path in sweep])}
for eta in (0.2, 0.7):
    long_runs[f"haw-q, η = {eta}"] = np.array([haw_q_empties(path, with_decay(UNCERTAIN, eta), k)
                                               for k, path in enumerate(sweep)])
print(pd.DataFrame({
    label: {"runs": emptied.size, "runs with an empty side": int(emptied.sum()), "fraction": emptied.mean()}
    for label, emptied in long_runs.items()
}).T.round(4).to_string())

No side of CKS or of haw-1 empties on these horizons.  One run of haw-q in 360 does, at
$\eta = 0.7$, and its path is dropped from that fan.  On the 48 paths of 1253 s none empties.  The
life of a side, about 13 400 s for haw-1 and 5 400 s for haw-q, is long against the horizons here,
and the assumption that no side empties holds on all but one run.